# Session 4: Scraping with Beautiful Soup

We will scrape **https://en.wikipedia.org/wiki/List_of_prime_ministers_of_India** and turn one webpage into a Pandas dataframe.

By the end, you should be able to:

- request a webpage
- check whether the request worked
- parse HTML
- identify a repeated container
- extract titles, prices, ratings, availability and links
- create and inspect a dataframe
- clean and save the scraped data

> Workflow: **Request → Parse → Select → Extract → Store → Validate**

## Before scraping

Understand the site's html structure.

## 1. Install packages if needed

### This cell installs the three external libraries used in the notebook:

- `requests` downloads the webpage.
- `beautifulsoup4` reads and searches the HTML.
- `pandas` turns the extracted information into a table.

In [ ]:
#!python3 -m pip install requests beautifulsoup4 pandas

## 2. Import libraries

### We are loading four tools:

- `requests` will fetch the webpage.
- `BeautifulSoup` will turn raw HTML into a searchable structure.
- `pandas`, shortened to `pd`, will create and analyse our dataframe.
- `urljoin` will turn incomplete links from the webpage into full URLs.

Running an import makes these tools available to the notebook.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

#### import requests
A Python library that sends requests to websites and downloads their content. Without it, Python has no way to ask a website for its contents.

#### from urllib.parse import urljoin
A function that combines a website's base URL with an incomplete (relative) link to create a full webpage address. We use it because many websites store links as relative paths instead of complete URLs.


Website
   │
   ▼
requests
(download page)
   │
   ▼
Beautiful Soup
(read HTML)
   │
   ▼
urljoin
(fix incomplete links)
   │
   ▼
Pandas
(store everything as a dataframe)

## 3. Store the URL

### We are saving the webpage address inside a variable called `url`.

The URL is text, so it must be placed inside quotation marks. Storing it in a variable means we can reuse the address later without typing it repeatedly.

The second line displays the value so we can confirm that the variable contains the expected address.

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_prime_ministers_of_India"
url

'https://en.wikipedia.org/wiki/List_of_prime_ministers_of_India'

## 4. Request the webpage

This is the first time Python communicates with a website.

When we write:

```python
response = requests.get(url)
```

the following happens:

1. Python sends an HTTP **GET request** to the website.
2. The website's server receives that request.
3. The server sends back a **response**.
4. We store that response in a variable called `response`.

Think of it like ordering food:

- **You** → make the request.
- **Restaurant** → prepares the order.
- **Delivery bag** → the response.

The response contains much more than the webpage itself. It also contains:
- the status code
- headers
- cookies
- and the HTML that generated the page.

We haven't downloaded a spreadsheet—we've downloaded the webpage itself.


### What we are about to do

This cell sends an HTTP request to the website.

- `requests.get(url, timeout=30)` asks the server for the page stored in `url`.
- `timeout=30` prevents Python from waiting forever if the website does not respond.
- The server's reply is saved in a variable called `response`.
- Writing `response` on the final line displays a short summary, usually something such as `<Response [200]>`.

At this stage, `response` contains the status code, headers and webpage HTML.

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/151.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=30)

response

<Response [200]>

In [4]:
response.status_code

200

### Optional: Check the response's status code.

A status code is the server's short message about what happened:

- `200` means the request succeeded.
- `403` means access was refused.
- `404` means the page was not found.
- `500` means the server encountered an error.

We should check this before trying to parse the page.

## 5. Look at the raw HTML

The browser turns HTML into something beautiful using CSS. But, Python doesn't see colours, buttons or layouts.
It only receives the underlying HTML.

The next two lines do two different jobs:

```python
html = response.text
```
stores the webpage's HTML inside a variable called `html`.

```python
print(html[:1000])
```

prints only the **first 1,000 characters**.

Why only the first thousand?

A webpage can contain tens of thousands of characters. Printing everything is a lot.

`[:1000]` is called **slicing** and simply means:

> "Show me the first thousand characters."

### Separate the webpage's HTML from the rest of the response.

- `response.text` contains the HTML as one long string of text.
- We store that string in a variable called `html`.
- `html[:1000]` uses slicing to keep only the first 1,000 characters.
- `print()` displays that shortened sample.

We inspect only the beginning because printing the full page would overwhelm the notebook.

In [5]:
html = response.text

print(html[:1000])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-feature-navigation-update-disabled vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="en" dir="ltr">
<head>
<meta charset="UTF-8">
<title>List of prime ministers of India - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned

In [6]:
html=response.text
print(html[10000:15000])

ropdown-checkbox "  aria-label="Main menu"   >
	<label id="vector-main-menu-dropdown-label" for="vector-main-menu-dropdown-checkbox" class="vector-dropdown-label cdx-button cdx-button--fake-button cdx-button--fake-button--enabled cdx-button--weight-quiet cdx-button--icon-only " aria-hidden="true"  ><span class="vector-icon mw-ui-icon-menu mw-ui-icon-wikimedia-menu cdx-button__icon"></span>

<span class="vector-dropdown-label-text">Main menu</span>
	</label>
	<div class="vector-dropdown-content">


				<div id="vector-main-menu-unpinned-container" class="vector-unpinned-container">
		
<div id="vector-main-menu" class="vector-main-menu vector-pinnable-element">
	<div
	class="vector-pinnable-header vector-main-menu-pinnable-header vector-pinnable-header-unpinned"
	data-feature-name="main-menu-pinned"
	data-pinnable-element-id="vector-main-menu"
	data-pinned-container-id="vector-main-menu-pinned-container"
	data-unpinned-container-id="vector-main-menu-unpinned-container"
>
	<div class="vec

Look for repeated structures such as:

```html
<article class="product_pod">
<p class="price_color">£51.77</p>
```

## 6. Parse the HTML

Right now `html` is just one very long string of text. Beautiful Soup converts that text into something we can search.

Think of it like this:

Before Beautiful Soup:
```
One giant wall of text
```

After Beautiful Soup:
```
A searchable tree of elements
```

Instead of searching through thousands of characters ourselves, we can now ask questions like:
- Find the first heading.
- Find every book.
- Find every price.
- Find every link.

### Turning the raw HTML string into a Beautiful Soup object.

- `html` is currently plain text.
- `"html.parser"` tells Beautiful Soup which parser to use.
- The parsed page is stored in `soup`.
- `type(soup)` checks what kind of Python object was created.

After this step, we can search the webpage by tags, classes and other HTML features.

In [7]:
soup = BeautifulSoup(html, "html.parser")

type(soup)

bs4.BeautifulSoup

## 7. Find the page heading

### What we are about to do

We are asking Beautiful Soup to find the first `<h1>` heading on the page.

- `soup.find("h1")` searches the parsed HTML.
- The matching HTML element is stored in `heading`.
- Displaying `heading` shows both the tag and its contents.

This is our first simple test that Beautiful Soup can locate an element successfully.

In [8]:
heading = soup.find("h1")

heading

<h1 class="firstHeading mw-first-heading" id="firstHeading"><span dir="ltr" lang="en"><span class="mw-page-title-main">List of prime ministers of India</span></span></h1>

In [9]:
print(heading.get_text(strip=True))

List of prime ministers of India


## 8. Find one book card

Every book lives inside:

```html
<article class="product_pod">
```

Every book has the **same HTML structure**. That repetition is exactly what makes scraping possible.
Rather than writing code twenty times, we teach Python how to recognise one book card. Then Python repeats the same process for every card.

### Locating the first repeated book card.

`article.product_pod` is a CSS selector:

- `article` refers to the HTML tag.
- The period means “class”.
- `product_pod` is the class name shared by every book card.
- `select_one()` returns only the first matching card.

We save that complete card in `first_book` so we can practise extracting one field at a time before looping over all books.

In [10]:
tables = soup.find_all("table")

len(tables)

10

In [11]:
for i, table in enumerate(tables):
    print(i, table.get_text(" ", strip=True)[:200])

0 Politics of India Constitution Amendment Basic structure doctrine Fundamental Rights, Directive Principles, and Fundamental Duties Human rights Judicial review Taxation Government President of India (
1 Portrait Name (born – died) Constituency Age when assumed office Term of office Duration in years and days Election Concurrent ministerial positions Party Ministry Head of State (Tenure) Jawaharlal Ne
2 Name Party Length of term Longest continuous term Total years of premiership Jawaharlal Nehru INC 16 years, 286 days Indira Gandhi INC/INC(I) / INC(R) 11 years, 59 days 15 years, 350 days Narendra Mod
3 Political parties by total timespan of their member holding PMO (as of 23 August 2026 ) No. Political party Number of Prime ministers Total years of holding PMO 1 INC/INC(I) / INC(R) 7 [ b ] 54 years,
4 v t e Prime Minister of India List By previous experience Prime ministers Jawaharlal Nehru Lal Bahadur Shastri Premiership Indira Gandhi Morarji Desai Premiership Charan Singh Rajiv Gand

In [12]:
pm_table = None

for table in tables:
    text = table.get_text(" ", strip=True)
    if "Age when assumed office" in text and "Term of office" in text:
        pm_table = table
        break

pm_table

<table class="wikitable plainrowheaders" id="mwzg" style="text-align:center">
<tbody id="mwzw"><tr id="mw0A"><th id="mw0Q">Portrait</th>
<th id="mw0g" style="width:15em">Name<br id="mw0w"/><small id="mw1A">(born<span about="#mwt95" data-mw='{"parts":[{"template":{"target":{"wt":"spaced ndash","href":"./Template:Spaced_ndash"},"params":{},"i":0}}]}' id="mw1Q" typeof="mw:Transclusion mw:Entity"> </span><span about="#mwt95" typeof="mw:Entity">–</span><span about="#mwt95" typeof="mw:Entity"> </span>died)<br id="mw1g"/>Constituency</small></th>
<th id="mw1w" style="width:6em">Age when assumed office</th>
<th colspan="3" id="mw2A">Term of office<br id="mw2Q"/><span about="#mwt96" data-mw='{"parts":[{"template":{"target":{"wt":"smaller","href":"./Template:Smaller"},"params":{"1":{"wt":"Duration in years and days"}},"i":0}}]}' id="mw2g" style="font-size: 85%;" typeof="mw:Transclusion">Duration in years and days</span></th>
<th id="mw2w">Election</th>
<th id="mw3A" style="width:23em">Concurrent

In [13]:
print(pm_table.get_text(" ", strip=True)[:1000])

Portrait Name (born – died) Constituency Age when assumed office Term of office Duration in years and days Election Concurrent ministerial positions Party Ministry Head of State (Tenure) Jawaharlal Nehru (1889–1964) MP for United Provinces (1947–1952) and Phulpur 57 years 274 days 15 August 1947 27 May 1964 [†] 16 years 286 days – External and Commonwealth Affairs Defence (1952–1955, 1957) Finance (1958) Indian National Congress Nehru I King George VI (1947–1950) Rajendra Prasad (1950–1962) 1951–52 Nehru II 1957 Nehru III 1962 Nehru IV Sarvepalli Radhakrishnan (1962–1967) Gulzarilal Nanda (acting) (1898–1998) MP for Sabarkantha 65 years, 328 days 27 May 1964 9 June 1964 [DIS] 13 days – Home Affairs External Affairs Atomic Energy Nanda I Lal Bahadur Shastri (1904–1966) MP for Allahabad 59 years 251 days 9 June 1964 11 January 1966 [†] 1 year, 216 days External Affairs (1964) Shastri Gulzarilal Nanda (acting) (1898–1998) MP for Sabarkantha 67 years, 191 days 11 January 1966 24 January 19

## 9. Extract one title

### Inside the first book card, locate the link that contains the title.

The selector `h3 a` means:

> Find an `<a>` link located inside an `<h3>` heading.

We store the matching HTML element in `title_element` and display it so we can inspect its visible text and attributes.

In [14]:
rows = pm_table.find_all("tr")

len(rows)

44

In [15]:
for row in rows[:5]:
    print(row.get_text(" | ", strip=True))

Portrait | Name | (born | – | died) | Constituency | Age when assumed office | Term of office | Duration in years and days | Election | Concurrent ministerial positions | Party | Ministry | Head of State | (Tenure)
Jawaharlal Nehru | (1889–1964) | MP for | United Provinces | (1947–1952) and | Phulpur | 57 years 274 days | 15 August | 1947 | 27 May | 1964 | [†] | 16 years 286 days | – | External and Commonwealth Affairs | Defence | (1952–1955, 1957) | Finance | (1958) | Indian National Congress | Nehru I | King George VI | (1947–1950)
Rajendra Prasad | (1950–1962)
1951–52 | Nehru II
1957 | Nehru III


### The full book title is stored in the link's `title` attribute.

- `title_element["title"]` extracts that attribute's value.
- We save it in a variable called `title`.
- The final line displays the extracted title.

This is different from `get_text()`: here we are reading an HTML attribute rather than the visible words between the tags.

- `.get_text()` extracts visible text.
- `["title"]` extracts the value of an HTML attribute.


In [16]:
first_row = rows[1]

cells = first_row.find_all(["th", "td"])

print(len(cells))

for i, cell in enumerate(cells):
    print(i, cell.get_text(" ", strip=True))

12
0 
1 Jawaharlal Nehru (1889–1964) MP for United Provinces (1947–1952) and Phulpur
2 57 years 274 days
3 15 August 1947
4 27 May 1964 [†]
5 16 years 286 days
6 –
7 External and Commonwealth Affairs Defence (1952–1955, 1957) Finance (1958)
8 
9 Indian National Congress
10 Nehru I
11 King George VI (1947–1950)


In [17]:
rows = pm_table.find_all("tr")

first_row = rows[1]

cells = first_row.find_all(["th", "td"])

for i, cell in enumerate(cells):
    print(i, cell.get_text(" ", strip=True))

0 
1 Jawaharlal Nehru (1889–1964) MP for United Provinces (1947–1952) and Phulpur
2 57 years 274 days
3 15 August 1947
4 27 May 1964 [†]
5 16 years 286 days
6 –
7 External and Commonwealth Affairs Defence (1952–1955, 1957) Finance (1958)
8 
9 Indian National Congress
10 Nehru I
11 King George VI (1947–1950)


In [18]:
name = cells[1].get_text(" ", strip=True)

name

'Jawaharlal Nehru (1889–1964) MP for United Provinces (1947–1952) and Phulpur'

### What we are about to do

The class list contains two items:

```python
["star-rating", "Three"]
```

Python counts list positions from zero:

- position `0` is `"star-rating"`
- position `1` is `"Three"`

This cell selects the second item and stores it as the book's rating.

## 13. Extract the link

### The webpage stores a relative link rather than a complete web address. `title_element.get("href")` retrieves that incomplete path.

`urljoin(url, relative_link)` combines the site's base URL with the relative path to create a complete, usable book URL.

In [20]:
constituency = cells[2].get_text(" ", strip=True)

constituency

'57 years 274 days'

In [21]:
age = cells[3].get_text(" ", strip=True)

age

'15 August 1947'

In [22]:
pm_row = {
    "name": name,
    "constituency": constituency,
    "age": age
}

pm_row

{'name': 'Jawaharlal Nehru (1889–1964) MP for United Provinces (1947–1952) and Phulpur',
 'constituency': '57 years 274 days',
 'age': '15 August 1947'}

The homepage shows 20 books, so we expect 20 matches.

In [23]:
data_rows = rows[1:]

len(data_rows)

43

In [24]:
for row in data_rows[:5]:
    print(row.get_text(" | ", strip=True))

Jawaharlal Nehru | (1889–1964) | MP for | United Provinces | (1947–1952) and | Phulpur | 57 years 274 days | 15 August | 1947 | 27 May | 1964 | [†] | 16 years 286 days | – | External and Commonwealth Affairs | Defence | (1952–1955, 1957) | Finance | (1958) | Indian National Congress | Nehru I | King George VI | (1947–1950)
Rajendra Prasad | (1950–1962)
1951–52 | Nehru II
1957 | Nehru III
1962 | Nehru IV | Sarvepalli Radhakrishnan | (1962–1967)


In [26]:
all_rows = []

for row in rows[1:]:
    cells = row.find_all(["th", "td"])

    if len(cells) < 8:
        continue

    all_rows.append({
        "name": cells[1].get_text(" ", strip=True),
        "constituency": cells[3].get_text(" ", strip=True),
        "age": cells[7].get_text(" ", strip=True)
    })

len(all_rows)

18

### Inspecting the first dictionary stored in the `rows` list.

Python uses zero-based indexing, so `rows[0]` means “show the first item”. This lets us confirm that the loop stored the expected fields before we create a dataframe.

In [27]:
len(all_rows)

18

In [28]:
all_rows[0]

{'name': 'Jawaharlal Nehru (1889–1964) MP for United Provinces (1947–1952) and Phulpur',
 'constituency': '15 August 1947',
 'age': 'External and Commonwealth Affairs Defence (1952–1955, 1957) Finance (1958)'}

## 18. Create a dataframe

This is the moment where web scraping meets Pandas.

Currently:

```
rows
```

is a **list of dictionaries**.

Pandas knows how to turn that structure into a table automatically.

Think about the mapping:

- one dictionary → one row
- dictionary keys → column names
- dictionary values → cells

After this line, everything you've already learned in Pandas works exactly the same.


### Convert the list of dictionaries into a Pandas dataframe.

Pandas interprets the structure automatically:

- each dictionary becomes one row
- each dictionary key becomes a column
- each dictionary value becomes a cell

`df.head()` then displays the first five scraped books so we can inspect the result.

In [29]:
df = pd.DataFrame(all_rows)

df.head()


,name,constituency,age
0,Jawaharlal Nehru (1889–1964) MP for United Pro...,15 August 1947,External and Commonwealth Affairs Defence (195...
1,Gulzarilal Nanda (acting) (1898–1998) MP for S...,27 May 1964,Home Affairs External Affairs Atomic Energy
2,Lal Bahadur Shastri (1904–1966) MP for Allahabad,9 June 1964,Shastri
3,Gulzarilal Nanda (acting) (1898–1998) MP for S...,11 January 1966,Nanda II
4,Indira Gandhi (1917–1984) MP for Uttar Pradesh...,24 January 1966,Indira I


### `df.shape` reports the dataframe's dimensions as:

```text
(number of rows, number of columns)
```

We expect 20 rows because the homepage contains 20 book cards. This is another check that our scraper found the expected number of observations.

In [30]:
df.shape

(18, 3)

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   name          18 non-null     str  
 1   constituency  18 non-null     str  
 2   age           18 non-null     str  
dtypes: str(3)
memory usage: 564.0 bytes


In [33]:
df["age"].head(10)

0    External and Commonwealth Affairs Defence (195...
1          Home Affairs External Affairs Atomic Energy
2                                              Shastri
3                                             Nanda II
4                                             Indira I
5        Finance (1977, 1979) Home Affairs (1978–1979)
6                                                 None
7          Defence (1980–1982) External Affairs (1984)
8    External Affairs (1984–1985, 1987–1988) Touris...
9    Defence Human Resource Development External Af...
Name: age, dtype: str

In [34]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

In [35]:
df["age"].head()

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: age, dtype: float64

## 21. Analyse the scraped data

In [36]:
df.sort_values("age", ascending=False)[
    ["name", "age"]
].head(10)

,name,age
0,Jawaharlal Nehru (1889–1964) MP for United Pro...,NaN
1,Gulzarilal Nanda (acting) (1898–1998) MP for S...,NaN
2,Lal Bahadur Shastri (1904–1966) MP for Allahabad,NaN
3,Gulzarilal Nanda (acting) (1898–1998) MP for S...,NaN
4,Indira Gandhi (1917–1984) MP for Uttar Pradesh...,NaN
5,Morarji Desai (1896–1995) MP for Surat,NaN
6,Charan Singh (1902–1987) MP for Baghpat,NaN
7,Indira Gandhi (1917–1984) MP for Medak,NaN
8,Rajiv Gandhi (1944–1991) MP for Amethi,NaN
9,V. P. Singh (1931–2008) MP for Fatehpur,NaN


In [37]:
df.sort_values("age")[
    ["name", "age"]
].head(10)

,name,age
0,Jawaharlal Nehru (1889–1964) MP for United Pro...,NaN
1,Gulzarilal Nanda (acting) (1898–1998) MP for S...,NaN
2,Lal Bahadur Shastri (1904–1966) MP for Allahabad,NaN
3,Gulzarilal Nanda (acting) (1898–1998) MP for S...,NaN
4,Indira Gandhi (1917–1984) MP for Uttar Pradesh...,NaN
5,Morarji Desai (1896–1995) MP for Surat,NaN
6,Charan Singh (1902–1987) MP for Baghpat,NaN
7,Indira Gandhi (1917–1984) MP for Medak,NaN
8,Rajiv Gandhi (1944–1991) MP for Amethi,NaN
9,V. P. Singh (1931–2008) MP for Fatehpur,NaN


### Count how many books received each rating.

- `value_counts()` counts the frequency of every rating value.
- `sort_index()` arranges the result in rating order from 1 to 5 rather than by frequency.

This shows the distribution of ratings on the first page.

In [39]:
df["age"].median()

np.float64(nan)

In [40]:
print("Rows:", len(df))

print("\nMissing values:")
print(df.isna().sum())

print("\nAge range:")
print(df["age"].min(), "to", df["age"].max())

Rows: 18

Missing values:
name             0
constituency     0
age             18
dtype: int64

Age range:
nan to nan


## 22. Validate the scrape

In [41]:
pm_df = df[
    [
        "name",
        "constituency",
        "age"
    ]
].copy()

In [42]:
pm_df.head()

,name,constituency,age
0,Jawaharlal Nehru (1889–1964) MP for United Pro...,15 August 1947,NaN
1,Gulzarilal Nanda (acting) (1898–1998) MP for S...,27 May 1964,NaN
2,Lal Bahadur Shastri (1904–1966) MP for Allahabad,9 June 1964,NaN
3,Gulzarilal Nanda (acting) (1898–1998) MP for S...,11 January 1966,NaN
4,Indira Gandhi (1917–1984) MP for Uttar Pradesh...,24 January 1966,NaN


A scraper can run without an error and still collect the wrong data. Compare a few rows manually with the webpage.


## 24. Save as CSV

In [43]:
output_filename = "prime_ministers_of_india.csv"

pm_df.to_csv(output_filename, index=False)

print(f"Saved {len(pm_df)} rows to {output_filename}")

Saved 18 rows to prime_ministers_of_india.csv


# Final recap

```text
URL
 ↓
requests.get()
 ↓
response.text
 ↓
BeautifulSoup()
 ↓
select repeated book cards
 ↓
extract fields
 ↓
list of dictionaries
 ↓
Pandas dataframe
 ↓
clean, validate and save
```